In [1]:
!pip install -q -U transformers accelerate
%pip install --no-cache-dir \
    "huggingface-hub==1.33.0" \
    "fsspec==2026.2.0"

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install num2words

Note: you may need to restart the kernel to use updated packages.


In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import base64
import os
import json
import os
from io import BytesIO
from PIL import Image

In [4]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU unavailable")

True
Tesla T4


In [5]:
%pip install --upgrade --force-reinstall "transformers==4.57.6" "huggingface_hub>=0.34.0,<1.0"

  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached filelock-4.2.1-py3-none-any.whl.metadata (2.0 kB)
  Using cached numpy-2.5.4-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached regex-2026.9.29-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.2 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached fsspec-20

In [6]:
%pip uninstall -y transformers
%pip install --no-cache-dir "transformers==4.57.6"

Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 740.9 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 75.5 MB/s eta 0:00:00 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [7]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_NAME = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"

processor_smol= AutoProcessor.from_pretrained(MODEL_NAME)

model_smol = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
).to("cuda")

model_smol.eval()

print("SmolVLM2 loaded successfully!")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

SmolVLM2 loaded successfully!


In [8]:
def query_from_path(image_path):
    # Helper to look up the correct query based on the image_path file name
    target_name = "images/" + os.path.basename(image_path)
    for record in annotations_list:
        if record["image_path"] == target_name:
            return record["question"]
    raise ValueError(f"No annotation record found for image path: {image_path}")

In [9]:
def get_base64_from_rgb(rgb_image):
    # Helper to convert in-memory numpy array (RGB) to base64 string
    bgr_img = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)
    _, buffer = cv2.imencode('.jpg', bgr_img)
    return base64.b64encode(buffer).decode('utf-8')

In [10]:
def evaluate_original(image_path):
  with open(image_path, "rb") as f:
    base64_image = base64.b64encode(f.read()).decode("utf-8")
  query_text = query_from_path(image_path)
  # Predict
  result = evaluate(query_text,base64_image)
  return result

In [11]:

def evaluate(query_text, b64_str):
    image_bytes = base64.b64decode(b64_str)
    image = Image.open(BytesIO(image_bytes)).convert("RGB")

    question = query_text + "\nAnswer only Yes or No."

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question}
            ]
        }
    ]

    text = processor_smol.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor_smol(
    text=[text],
    images=[image],
    padding=True,
    return_tensors="pt"
    ).to(model_smol.device)

    with torch.inference_mode():
        output_ids = model_smol.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    generated_ids = output_ids[:, len(inputs["input_ids"][0]):]

    answer = processor_smol.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return answer.strip()



In [12]:
import transformers
print(transformers.__version__)

4.57.6


In [20]:
import os
import pandas as pd
from tqdm.notebook import tqdm

# --- Paths Configuration (Adjust as needed) ---
IMAGE_DIR = "/kaggle/input/datasets/ramakanthjogada/true-positive-only/repope_yes_500_unique/images"
ANNOTATIONS_PATH = "/kaggle/input/datasets/ramakanthjogada/true-positive-only/repope_yes_500_unique/annotations.json"

# Load annotations lookup map for faster access
with open(ANNOTATIONS_PATH, "r") as f:
    annotations_list = json.load(f)


# Filter out records where the query is "Is there a person in the image?"
filtered_annotations = [
    record for record in annotations_list
    if record.get("question", "").strip() != "Is there a person in the image?"
]

# Create a lookup map for ground truth from the filtered records
ground_truth_map = {}
for record in filtered_annotations:
    if record.get("image_path", "").startswith("images/"):
        img_id = record["image_id"]
        ground_truth_map[int(img_id)] = record["label"]

# Get list of all images in the directory
all_image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.jpeg', '.png'))]

# We only want to process images that actually exist in our filtered ground truth map
image_files = []
for filename in all_image_files:
    try:
        image_id = int(os.path.splitext(filename)[0])
        if image_id in ground_truth_map:
            image_files.append(filename)
    except ValueError:
        continue

print(f"Total raw annotations: {len(annotations_list)}")
print(f"Filtered annotations (excluding 'person' queries): {len(filtered_annotations)}")
print(f"Final count of rows to evaluate (matching image files): {len(image_files)}")


results = []

print(f"Processing {len(image_files)} images through all variations...")
for filename in tqdm(image_files):
    # Extract image_id from filename (e.g., '000000046207.jpg' -> 46207)
    try:
        image_id = int(os.path.splitext(filename)[0])
    except ValueError:
        continue

    image_path = os.path.join(IMAGE_DIR, filename)
    ground_truth = ground_truth_map.get(image_id, "Unknown")

    # Evaluate under all conditions
    try:
        orig_val = evaluate_original(image_path)
    except Exception as e:
        orig_val = f"Error: {str(e)}"
         
        
        
    results.append({
        "Image_Id": image_id,
        "Ground_truth": ground_truth,
        "Original": orig_val})
    

df_results = pd.DataFrame(results)
display(df_results.head(10))

Total raw annotations: 491
Filtered annotations (excluding 'person' queries): 315
Final count of rows to evaluate (matching image files): 315
Processing 315 images through all variations...


  0%|          | 0/315 [00:00<?, ?it/s]

,Image_Id,Ground_truth,Original
0,102439,yes,Yes.
1,108189,yes,Yes.
2,579231,yes,No.
3,75591,yes,Yes.
4,514248,yes,Yes.
5,516601,yes,No.
6,526321,yes,Yes.
7,207205,yes,Yes.
8,280734,yes,Yes.
9,468997,yes,No.


In [21]:
df_results

,Image_Id,Ground_truth,Original
0,102439,yes,Yes.
1,108189,yes,Yes.
2,579231,yes,No.
3,75591,yes,Yes.
4,514248,yes,Yes.
...,...,...,...
310,17379,yes,Yes.
311,399702,yes,Yes.
312,418680,yes,Yes.
313,172648,yes,Yes.


In [25]:
df_results.rename(columns={"Original": "Smol-2.2B"}, inplace=True)

In [26]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/ramakanthjogada/diff-models-evals-on-pope/diff_models_eval (1).csv", usecols=["Image_Id","Ground_truth","Smol-2.2B"])
df

,Image_Id,Ground_truth,Smol-2.2B
0,6578,No,Yes.
1,8602,No,No.
2,9469,No,No.
3,9712,No,No.
4,10733,No,No.
...,...,...,...
495,574356,No,No.
496,574837,No,No.
497,575121,No,No.
498,575916,No,No.


In [27]:
append_df = pd.concat([df,df_results], ignore_index=True)

In [28]:
append_df

,Image_Id,Ground_truth,Smol-2.2B
0,6578,No,Yes.
1,8602,No,No.
2,9469,No,No.
3,9712,No,No.
4,10733,No,No.
...,...,...,...
810,17379,yes,Yes.
811,399702,yes,Yes.
812,418680,yes,Yes.
813,172648,yes,Yes.


In [29]:
append_df = append_df.drop_duplicates(subset=["Image_Id"], keep="first")
append_df

,Image_Id,Ground_truth,Smol-2.2B
0,6578,No,Yes.
1,8602,No,No.
2,9469,No,No.
3,9712,No,No.
4,10733,No,No.
...,...,...,...
810,17379,yes,Yes.
811,399702,yes,Yes.
812,418680,yes,Yes.
813,172648,yes,Yes.


In [30]:
append_df.to_csv("Smol-2.2B-without-person.csv")